In [1]:
import torch
import torch.nn.functional as F 
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [4]:
#building dataset

block_size = 3 #context length
X,Y = [],[]
for w in words[:5]:
   print(w)
   context = [0] * block_size
   for ch in w + '.' :
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print (''.join(itos[i] for i in context), '-->', itos[ix])
    context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)



emma
... --> e
..e --> m
.em --> m
emm --> a
mma --> .
olivia
... --> o
..o --> l
.ol --> i
oli --> v
liv --> i
ivi --> a
via --> .
ava
... --> a
..a --> v
.av --> a
ava --> .
isabella
... --> i
..i --> s
.is --> a
isa --> b
sab --> e
abe --> l
bel --> l
ell --> a
lla --> .
sophia
... --> s
..s --> o
.so --> p
sop --> h
oph --> i
phi --> a
hia --> .


In [5]:
C = torch.randn((27,2))

In [6]:
emb = C[X]

In [7]:
C[X][13,2]

tensor([ 0.3864, -0.1563])

In [8]:
C[1]

tensor([ 0.3864, -0.1563])

In [9]:
w1 = torch.randn((6,100))
b1 = torch.randn(100)

In [10]:
h = torch.tanh(emb.view(-1,6) @ w1 + b1)

In [11]:
h

tensor([[ 0.7561, -0.9841, -0.7452,  ..., -0.1884,  0.9859,  0.3546],
        [ 0.3266, -0.9847, -0.8626,  ...,  0.3886,  0.9988,  0.4240],
        [ 0.9803, -0.9235, -0.5714,  ..., -0.5770,  0.9200,  0.2233],
        ...,
        [ 1.0000, -0.9979,  0.6667,  ..., -0.7672,  0.9587, -0.4311],
        [-0.4098,  0.9997, -0.2683,  ...,  0.9901,  1.0000,  0.8567],
        [ 0.4956, -0.9915, -0.9985,  ..., -0.9997,  0.9996, -0.5617]])

In [12]:
h.shape

torch.Size([32, 100])

In [13]:
w2 = torch.randn([100,27])
b2 = torch.randn(27)

In [14]:
logits = h @ w2 + b2

In [15]:
logits.shape

torch.Size([32, 27])

In [16]:
logits

tensor([[-5.5371e+00, -1.1299e+01,  3.8839e+00, -1.3220e+01,  5.8139e+00,
          1.7046e+01,  1.4439e+01,  9.2498e+00, -8.4553e+00,  1.6371e+01,
         -1.0832e+01,  7.4055e+00,  4.3627e+00, -3.6665e+00, -6.9283e+00,
         -4.4314e+00, -2.1967e+00, -8.0767e-01, -4.4895e+00, -1.1296e+00,
          7.4017e+00, -1.4633e+01,  4.8067e-01,  9.4782e-01,  4.6099e+00,
         -3.5431e+00,  9.2110e+00],
        [-4.2931e+00, -1.0736e+01,  7.1752e+00, -1.3087e+01, -5.6765e+00,
          1.5040e+01,  3.0517e+00,  9.4511e+00, -6.3635e+00,  1.4037e+01,
         -5.7281e+00,  8.8728e+00,  7.3485e+00, -2.2264e+00, -8.3471e+00,
          5.0022e-01, -9.3034e+00,  9.8948e+00, -1.7900e+00, -7.1253e+00,
          6.4622e+00, -7.1168e+00,  6.2279e-01, -3.3129e+00,  1.6228e+00,
         -2.6720e+00,  2.9445e+00],
        [-1.1158e+01, -9.6440e+00,  1.1474e+01, -1.1549e+01,  1.0537e+01,
          8.6186e+00,  1.7685e+01,  3.7995e+00, -1.0022e+01,  1.8130e+01,
         -1.2173e+01, -2.6701e-01,  6.09

In [17]:
counts = logits.exp()

In [18]:
prob = counts/counts.sum(1, keepdims=True)

In [19]:
prob.shape

torch.Size([32, 27])

In [20]:
torch.arange(32)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31])

In [21]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [ ]:
loss = -prob[torch.arange(32), Y].log().mean()


In [25]:
loss

tensor(17.7362)

In [26]:
F.cross_entropy(logits, Y)

tensor(17.7362)